In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('shop.db')

df_orders    = pd.read_sql_query("SELECT * FROM orders", conn)
df_customers = pd.read_sql_query("SELECT * FROM customers", conn)
df_items     = pd.read_sql_query("SELECT * FROM order_items", conn)
df_shipments = pd.read_sql_query("SELECT * FROM shipments", conn)
df_products  = pd.read_sql_query("SELECT * FROM products", conn)


In [16]:
# Aggregate order_items per order
items_agg = df_items.groupby('order_id').agg(
    item_count=('order_item_id', 'count'),
    total_quantity=('quantity', 'sum'),
    avg_unit_price=('unit_price', 'mean')
).reset_index()

# fills in null promo codes with 'none'
df_orders['promo_code'] = df_orders['promo_code'].fillna('none')

# converts order_datetime and birthdate to datetime objects
df_orders['order_datetime'] = pd.to_datetime(df_orders['order_datetime'])
df_customers['birthdate']   = pd.to_datetime(df_customers['birthdate'])

# Build master DataFrame
df = df_orders \
    .merge(df_customers, on='customer_id', how='left') \
    .merge(items_agg, on='order_id', how='left') \
    .merge(df_shipments[['order_id','carrier','shipping_method','distance_band','late_delivery']], on='order_id', how='left')

In [17]:
df.head()

,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,...,customer_segment,loyalty_tier,is_active,item_count,total_quantity,avg_unit_price,carrier,shipping_method,distance_band,late_delivery
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,...,standard,silver,1,5,9,69.242,UPS,expedited,regional,1
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,...,standard,silver,1,5,7,133.300,FedEx,expedited,local,1
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,...,standard,silver,1,3,5,140.850,FedEx,expedited,national,1
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,...,standard,silver,1,1,1,137.600,UPS,standard,regional,0
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,...,standard,silver,1,1,1,17.070,USPS,standard,regional,1
